# GNPS Mass spectrometry Run Identifiers (MRIs) Analysis

This notebook downloads, processes, and visualises the `gnps_mris.tsv` file from the
[GeMS (GNPS Experimental Mass Spectra)](https://huggingface.co/datasets/roman-bushuiev/GeMS) dataset.

The file contains **Universal Spectrum Identifiers (USIs)** pointing to mass spectrometry run files
(`.mzML` / `.mzXML`) hosted in the [MassIVE/GNPS](https://massive.ucsd.edu/) repository. Each USI
follows the format `mzspec:<DATASET_ID>:<FILE_PATH>`, identifying a specific LC-MS/MS run from which
the unannotated spectra in GeMS were mined.

**Sections:**
1. Download and load
2. USI parsing and structure
3. MassIVE dataset distribution
4. File format analysis
5. Path/collection type analysis
6. Temporal patterns (dataset submission dates)
7. Top datasets deep-dive
8. Summary statistics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from urllib.parse import unquote
from collections import Counter

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

## 1. Download and Load

In [ ]:
DATA_URL = "https://huggingface.co/datasets/roman-bushuiev/GeMS/resolve/main/data/auxiliary/gnps_mris.tsv"
LOCAL_PATH = Path("../data/gnps_mris.tsv")

if not LOCAL_PATH.exists():
    LOCAL_PATH.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {DATA_URL} ...")
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, LOCAL_PATH)
    print(f"Saved to {LOCAL_PATH} ({LOCAL_PATH.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Already downloaded: {LOCAL_PATH} ({LOCAL_PATH.stat().st_size / 1e6:.1f} MB)")

df = pd.read_csv(LOCAL_PATH, sep="\t")
print(f"\nRows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head(10)

## 2. USI Parsing and Structure

Each USI follows the pattern `mzspec:<MSV_DATASET_ID>:<FILE_PATH>`. We extract:
- **dataset_id**: The MassIVE accession (e.g. `MSV000086270`)
- **file_path**: Path to the run file within the dataset
- **file_ext**: File format (`.mzML` or `.mzXML`)
- **collection_type**: Top-level directory in the path (e.g. `ccms_peak`, `peak`, `updates`)

In [ ]:
# Parse USIs into structured columns
parsed = df["usi"].str.extract(r"mzspec:(?P<dataset_id>MSV\d+):(?P<file_path>.+)")
df = pd.concat([df, parsed], axis=1)

# URL-decode file paths
df["file_path"] = df["file_path"].apply(unquote)

# Extract file extension (normalised to lowercase)
df["file_ext"] = df["file_path"].str.extract(r"(\.[^.]+)$", expand=False).str.lower()

# Extract collection type (first path segment)
df["collection_type"] = df["file_path"].str.extract(r"^([^/]+)/", expand=False)

# Extract filename
df["filename"] = df["file_path"].str.rsplit("/", n=1).str[-1]

print(f"Parsed {len(df):,} USIs")
print(f"Unique datasets: {df['dataset_id'].nunique():,}")
print(f"Unique file paths: {df['file_path'].nunique():,}")
print(f"\nSample parsed rows:")
df[["dataset_id", "collection_type", "file_ext", "filename"]].head(8)

In [ ]:
# Check for duplicates
n_dupes = df["usi"].duplicated().sum()
print(f"Duplicate USIs: {n_dupes:,} ({n_dupes / len(df) * 100:.2f}%)")

# Check for parsing failures
n_unparsed = df["dataset_id"].isna().sum()
print(f"Unparsed USIs: {n_unparsed}")

## 3. MassIVE Dataset Distribution

How are the 338k mass spec runs distributed across MassIVE datasets?

In [ ]:
dataset_counts = df["dataset_id"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Log-scale histogram of runs per dataset
axes[0].hist(dataset_counts.values, bins=np.logspace(0, np.log10(dataset_counts.max()), 50),
             color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_xscale("log")
axes[0].set_xlabel("Runs per dataset")
axes[0].set_ylabel("Number of datasets")
axes[0].set_title(f"Distribution of runs per MassIVE dataset (n={len(dataset_counts):,})")
axes[0].axvline(dataset_counts.median(), color="red", ls="--",
                label=f"Median = {dataset_counts.median():.0f}")
axes[0].axvline(dataset_counts.mean(), color="orange", ls="--",
                label=f"Mean = {dataset_counts.mean():.0f}")
axes[0].legend()

# Cumulative coverage: fraction of runs covered by top-N datasets
cumulative = dataset_counts.values.cumsum() / dataset_counts.values.sum()
axes[1].plot(range(1, len(cumulative) + 1), cumulative * 100, color="steelblue", lw=2)
axes[1].set_xlabel("Number of datasets (ranked by size)")
axes[1].set_ylabel("Cumulative % of runs")
axes[1].set_title("Cumulative coverage by top datasets")
axes[1].axhline(50, color="red", ls=":", alpha=0.5, label="50%")
axes[1].axhline(90, color="orange", ls=":", alpha=0.5, label="90%")

# Find how many datasets needed for 50% and 90%
n_50 = np.searchsorted(cumulative, 0.5) + 1
n_90 = np.searchsorted(cumulative, 0.9) + 1
axes[1].axvline(n_50, color="red", ls=":", alpha=0.5)
axes[1].axvline(n_90, color="orange", ls=":", alpha=0.5)
axes[1].annotate(f"{n_50} datasets = 50%", (n_50, 50), fontsize=10,
                 xytext=(n_50 + 50, 45), arrowprops=dict(arrowstyle="->"))
axes[1].annotate(f"{n_90} datasets = 90%", (n_90, 90), fontsize=10,
                 xytext=(n_90 + 100, 85), arrowprops=dict(arrowstyle="->"))
axes[1].legend()

fig.tight_layout()
plt.show()

print(f"\nSummary statistics (runs per dataset):")
print(dataset_counts.describe().to_string())

In [ ]:
# Top 20 datasets by number of runs
top20 = dataset_counts.head(20)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(range(len(top20)), top20.values, color="steelblue", edgecolor="white")
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Number of runs")
ax.set_title("Top 20 MassIVE datasets by run count")

# Add count labels
for bar, val in zip(bars, top20.values):
    ax.text(val + 100, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=9)

pct = top20.sum() / len(df) * 100
ax.text(0.98, 0.02, f"Top 20 = {top20.sum():,} runs ({pct:.1f}%)",
        transform=ax.transAxes, ha="right", fontsize=11, style="italic")

fig.tight_layout()
plt.show()

## 4. File Format Analysis

In [ ]:
ext_counts = df["file_ext"].value_counts()
print("File format distribution:")
for ext, count in ext_counts.items():
    print(f"  {ext:10s} {count:>8,}  ({count / len(df) * 100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
# Merge case variants
format_map = {"mzml": "mzML", "mzxml": "mzXML"}
df["format"] = df["file_ext"].str.lstrip(".").str.lower().map(format_map).fillna("other")
fmt_counts = df["format"].value_counts()

colors = ["#4C72B0", "#DD8452", "#55A868"]
wedges, texts, autotexts = axes[0].pie(
    fmt_counts.values, labels=fmt_counts.index, autopct="%1.1f%%",
    colors=colors[:len(fmt_counts)], startangle=90,
    textprops={"fontsize": 12}
)
axes[0].set_title("File format distribution")

# Format by dataset: are some datasets mzXML-only?
fmt_by_dataset = df.groupby("dataset_id")["format"].apply(
    lambda x: "mixed" if x.nunique() > 1 else x.iloc[0]
).value_counts()

axes[1].bar(fmt_by_dataset.index, fmt_by_dataset.values, color=colors[:len(fmt_by_dataset)])
axes[1].set_ylabel("Number of datasets")
axes[1].set_title("File format per dataset")
for i, (label, val) in enumerate(fmt_by_dataset.items()):
    axes[1].text(i, val + 10, str(val), ha="center", fontsize=11)

fig.tight_layout()
plt.show()

## 5. Collection Type Analysis

The first path segment in each USI indicates the collection type within MassIVE:
- `ccms_peak`: CCMS-processed peak lists
- `peak`: User-uploaded peak files  
- `raw`: Raw instrument data
- `updates`: Updated/versioned submissions
- `spectrum`: Individual spectrum files

In [ ]:
coll_counts = df["collection_type"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of collection types
ax = axes[0]
bars = ax.bar(range(len(coll_counts)), coll_counts.values, color="steelblue", edgecolor="white")
ax.set_xticks(range(len(coll_counts)))
ax.set_xticklabels(coll_counts.index, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Number of runs")
ax.set_title("Collection type distribution")
ax.set_yscale("log")
for bar, val in zip(bars, coll_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val * 1.15,
            f"{val:,}", ha="center", fontsize=9)

# Collection type by format
ct_fmt = pd.crosstab(df["collection_type"], df["format"])
ct_fmt_top = ct_fmt.loc[coll_counts.head(6).index]
ct_fmt_top.plot.barh(stacked=True, ax=axes[1], color=colors[:ct_fmt_top.shape[1]])
axes[1].set_xlabel("Number of runs")
axes[1].set_title("Collection type by file format")
axes[1].legend(title="Format")

fig.tight_layout()
plt.show()

## 6. Temporal Patterns

Some USI paths include update timestamps (e.g. `updates/2022-05-03_...`). We can extract these
to understand the temporal distribution of data submissions.

In [ ]:
# Extract dates from update paths
date_pattern = r"updates/(\d{4}-\d{2}-\d{2})"
df["update_date"] = df["file_path"].str.extract(date_pattern, expand=False)
df["update_date"] = pd.to_datetime(df["update_date"], errors="coerce")

has_date = df["update_date"].notna()
print(f"Runs with extractable update date: {has_date.sum():,} ({has_date.mean() * 100:.1f}%)")

if has_date.sum() > 0:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    # Timeline of updates
    monthly = df.loc[has_date].set_index("update_date").resample("ME").size()
    axes[0].bar(monthly.index, monthly.values, width=25, color="steelblue", alpha=0.8)
    axes[0].set_xlabel("Date")
    axes[0].set_ylabel("Number of runs")
    axes[0].set_title("Monthly distribution of dataset updates")

    # Year distribution
    yearly = df.loc[has_date, "update_date"].dt.year.value_counts().sort_index()
    axes[1].bar(yearly.index, yearly.values, color="steelblue", edgecolor="white")
    axes[1].set_xlabel("Year")
    axes[1].set_ylabel("Number of runs")
    axes[1].set_title("Yearly distribution of dataset updates")
    for x, y in zip(yearly.index, yearly.values):
        axes[1].text(x, y + 200, f"{y:,}", ha="center", fontsize=10)

    fig.tight_layout()
    plt.show()

    print(f"\nDate range: {df.loc[has_date, 'update_date'].min().date()} to "
          f"{df.loc[has_date, 'update_date'].max().date()}")

## 7. Top Datasets Deep-Dive

Examine the internal structure of the largest contributing datasets.

In [ ]:
top5_ids = dataset_counts.head(5).index.tolist()

fig, axes = plt.subplots(1, len(top5_ids), figsize=(4 * len(top5_ids), 5), sharey=False)

for ax, dsid in zip(axes, top5_ids):
    subset = df[df["dataset_id"] == dsid]
    
    # Get subdirectory structure (2nd path level)
    sub_dirs = subset["file_path"].str.split("/").str[:2].str.join("/")
    top_subdirs = sub_dirs.value_counts().head(8)
    
    ax.barh(range(len(top_subdirs)), top_subdirs.values, color="steelblue", alpha=0.8)
    ax.set_yticks(range(len(top_subdirs)))
    labels = [s if len(s) < 30 else s[:27] + "..." for s in top_subdirs.index]
    ax.set_yticklabels(labels, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("Runs")
    ax.set_title(f"{dsid}\n({len(subset):,} runs)", fontsize=10)

fig.suptitle("Subdirectory structure of top 5 datasets", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Filename patterns: look for common naming conventions
# Extract likely sample type indicators from filenames
keywords = ["blank", "qc", "std", "pool", "neg", "pos", "ms1", "ms2", "dda", "dia", "swath"]

keyword_counts = {}
for kw in keywords:
    mask = df["filename"].str.lower().str.contains(kw, na=False)
    keyword_counts[kw] = mask.sum()

kw_series = pd.Series(keyword_counts).sort_values(ascending=True)
kw_series = kw_series[kw_series > 0]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(kw_series.index, kw_series.values, color="steelblue", edgecolor="white")
ax.set_xlabel("Number of runs")
ax.set_title("Filename keyword frequency (case-insensitive)")
for bar, val in zip(bars, kw_series.values):
    ax.text(val + 100, bar.get_y() + bar.get_height() / 2,
            f"{val:,} ({val / len(df) * 100:.1f}%)", va="center", fontsize=9)

fig.tight_layout()
plt.show()

print("\nNotable observations:")
n_blank = keyword_counts["blank"]
n_qc = keyword_counts["qc"]
print(f"  - {n_blank:,} runs ({n_blank/len(df)*100:.1f}%) contain 'blank' (likely blank injections)")
print(f"  - {n_qc:,} runs ({n_qc/len(df)*100:.1f}%) contain 'qc' (quality control samples)")
print(f"  - Polarity indicators: {keyword_counts['neg']:,} neg, {keyword_counts['pos']:,} pos")
print(f"  - Acquisition modes: {keyword_counts['dda']:,} DDA, {keyword_counts['dia']:,} DIA, {keyword_counts['swath']:,} SWATH")

## 8. Dataset Size Distribution and Diversity

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (0,0) Dataset size CDF
ax = axes[0, 0]
sorted_counts = np.sort(dataset_counts.values)[::-1]
cdf = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)
ax.plot(sorted_counts, cdf, color="steelblue", lw=2)
ax.set_xscale("log")
ax.set_xlabel("Runs per dataset (log scale)")
ax.set_ylabel("Fraction of datasets with >= X runs")
ax.set_title("Dataset size CDF (complementary)")
ax.axhline(0.5, color="red", ls=":", alpha=0.5)

# (0,1) Lorenz curve (inequality of dataset contributions)
ax = axes[0, 1]
sorted_asc = np.sort(dataset_counts.values)
cumshare = sorted_asc.cumsum() / sorted_asc.sum()
pop_share = np.arange(1, len(sorted_asc) + 1) / len(sorted_asc)
ax.plot(pop_share, cumshare, color="steelblue", lw=2, label="Lorenz curve")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect equality")
ax.fill_between(pop_share, cumshare, pop_share, alpha=0.15, color="steelblue")

# Gini coefficient
gini = 1 - 2 * np.trapz(cumshare, pop_share)
ax.set_xlabel("Fraction of datasets")
ax.set_ylabel("Fraction of total runs")
ax.set_title(f"Lorenz curve (Gini = {gini:.3f})")
ax.legend()

# (1,0) Unique filenames per dataset
ax = axes[1, 0]
files_per_ds = df.groupby("dataset_id")["filename"].nunique()
ax.hist(files_per_ds.values, bins=np.logspace(0, np.log10(files_per_ds.max()), 40),
        color="steelblue", edgecolor="white", alpha=0.8)
ax.set_xscale("log")
ax.set_xlabel("Unique filenames per dataset")
ax.set_ylabel("Number of datasets")
ax.set_title("Filename diversity per dataset")

# (1,1) Average filename length distribution
ax = axes[1, 1]
fname_lengths = df["filename"].str.len()
ax.hist(fname_lengths, bins=60, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(fname_lengths.median(), color="red", ls="--",
           label=f"Median = {fname_lengths.median():.0f} chars")
ax.set_xlabel("Filename length (characters)")
ax.set_ylabel("Count")
ax.set_title("Filename length distribution")
ax.legend()

fig.tight_layout()
plt.show()

## 9. MassIVE Dataset ID Numerical Distribution

The MassIVE accession numbers (MSV000XXXXX) are sequential. Their distribution reveals
which "eras" of MassIVE deposits contributed most to GeMS.

In [ ]:
# Extract numeric part of dataset IDs
df["dataset_num"] = df["dataset_id"].str.extract(r"MSV(\d+)", expand=False).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of dataset IDs (weighted by run count)
axes[0].hist(df["dataset_num"], bins=80, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("MassIVE accession number")
axes[0].set_ylabel("Number of runs")
axes[0].set_title("Run distribution across MassIVE accessions")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f"MSV{int(x):09d}"))
axes[0].tick_params(axis="x", rotation=30)

# Unique datasets histogram (unweighted)
unique_nums = df.drop_duplicates("dataset_id")["dataset_num"]
axes[1].hist(unique_nums, bins=60, color="steelblue", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("MassIVE accession number")
axes[1].set_ylabel("Number of unique datasets")
axes[1].set_title("Dataset accession number distribution")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f"MSV{int(x):09d}"))
axes[1].tick_params(axis="x", rotation=30)

fig.tight_layout()
plt.show()

print(f"Accession range: MSV{unique_nums.min():09d} to MSV{unique_nums.max():09d}")

## 10. Summary

In [ ]:
print("=" * 60)
print("GNPS MRIs Summary")
print("=" * 60)
print(f"Total mass spec runs:     {len(df):>10,}")
print(f"Unique MassIVE datasets:  {df['dataset_id'].nunique():>10,}")
print(f"Unique file paths:        {df['file_path'].nunique():>10,}")
print(f"Duplicate USIs:           {df['usi'].duplicated().sum():>10,}")
print()
print("File formats:")
for fmt, count in fmt_counts.items():
    print(f"  {fmt:>8s}: {count:>8,} ({count / len(df) * 100:.1f}%)")
print()
print("Runs per dataset:")
print(f"  Mean:   {dataset_counts.mean():>10.1f}")
print(f"  Median: {dataset_counts.median():>10.1f}")
print(f"  Max:    {dataset_counts.max():>10,}  ({dataset_counts.idxmax()})")
print(f"  Min:    {dataset_counts.min():>10,}")
print()
print(f"Gini coefficient:         {gini:>10.3f}  (0=equal, 1=concentrated)")
print(f"Top 10 datasets cover:    {dataset_counts.head(10).sum() / len(df) * 100:>9.1f}%")
print(f"Top 100 datasets cover:   {dataset_counts.head(100).sum() / len(df) * 100:>9.1f}%")
print()
print("Collection types:")
for coll, count in coll_counts.head(6).items():
    print(f"  {coll:>15s}: {count:>8,} ({count / len(df) * 100:.1f}%)")
print("=" * 60)